# AI Code Quality Auditor — five-condition statistical comparison

Loads one or more experiment reports from `data/reports/*.csv`, then for each metric runs:

1. Per-condition descriptive stats (n, mean, median, IQR).
2. **Shapiro–Wilk** normality test per condition.
3. **Kruskal–Wallis** omnibus test across the five conditions (non-parametric — appropriate for small n and non-normal distributions, which is the realistic case for this instrument).
4. **Pairwise Mann–Whitney U** with Bonferroni correction when Kruskal–Wallis is significant at α = 0.05.
5. Per-metric boxplot.

The MSc proposal (§5) frames this as the analytical layer; this notebook is the operational form of that layer.

In [ ]:
import glob
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

REPORTS_DIR = Path('../data/reports')
ALPHA = 0.05
CONDITIONS = ['human_control', 'claude_code', 'cursor_agent', 'antigravity', 'replit_agent']

## 1. Load every CSV in `data/reports/`

Each CSV is the output of one `auditor experiment` invocation — one row per `(run_id, condition, metric)`. Concatenating across runs gives us the sample size needed for inferential tests.

In [ ]:
csv_files = sorted(glob.glob(str(REPORTS_DIR / '*.csv')))
assert csv_files, f'No CSVs in {REPORTS_DIR.resolve()} — run `auditor experiment` first.'
df = pd.concat([pd.read_csv(p) for p in csv_files], ignore_index=True)
df['value'] = pd.to_numeric(df['value'])
print(f'Loaded {len(csv_files)} report(s), {len(df)} rows, {df.run_id.nunique()} run_ids.')
df.head()

## 2. Descriptive stats per (metric, condition)

In [ ]:
desc = (df.groupby(['metric', 'condition'])['value']
          .agg(n='count', mean='mean', median='median',
               q1=lambda s: s.quantile(0.25),
               q3=lambda s: s.quantile(0.75),
               std='std')
          .round(3))
desc

## 3. Normality (Shapiro–Wilk) per (metric, condition)

Requires n ≥ 3 per group. Where the experiment has only one run per condition, normality is undefined and we fall back to non-parametric tests by default anyway.

In [ ]:
rows = []
for (metric, cond), sub in df.groupby(['metric', 'condition']):
    if sub['value'].nunique() < 2 or len(sub) < 3:
        rows.append({'metric': metric, 'condition': cond, 'n': len(sub), 'p': float('nan'), 'normal_at_0.05': None})
        continue
    p = stats.shapiro(sub['value']).pvalue
    rows.append({'metric': metric, 'condition': cond, 'n': len(sub), 'p': round(p, 4), 'normal_at_0.05': p > ALPHA})
pd.DataFrame(rows)

## 4. Kruskal–Wallis omnibus per metric

Tests **H0: distributions across the five conditions are identical**.
If rejected at α = 0.05, follow up with pairwise Mann–Whitney + Bonferroni.

In [ ]:
omnibus = []
for metric, sub in df.groupby('metric'):
    groups = [g['value'].values for _, g in sub.groupby('condition') if len(g) > 0]
    if len(groups) < 2 or any(len(g) < 1 for g in groups):
        omnibus.append({'metric': metric, 'H': None, 'p': None, 'significant': None,
                        'note': 'insufficient groups'})
        continue
    if all(len(set(g)) == 1 for g in groups) and len({g[0] for g in groups}) == 1:
        omnibus.append({'metric': metric, 'H': 0.0, 'p': 1.0, 'significant': False,
                        'note': 'all values identical'})
        continue
    try:
        H, p = stats.kruskal(*groups)
    except ValueError as e:
        omnibus.append({'metric': metric, 'H': None, 'p': None, 'significant': None, 'note': str(e)})
        continue
    omnibus.append({'metric': metric, 'H': round(H, 3), 'p': round(p, 4),
                    'significant': p < ALPHA, 'note': ''})
omnibus_df = pd.DataFrame(omnibus)
omnibus_df

## 5. Pairwise Mann–Whitney with Bonferroni

Only run for metrics flagged significant by Kruskal–Wallis.

In [ ]:
pairs = []
sig_metrics = omnibus_df.query('significant == True')['metric'].tolist() if not omnibus_df.empty else []
for metric in sig_metrics:
    sub = df[df['metric'] == metric]
    conds = sorted(sub['condition'].unique())
    n_pairs = len(list(combinations(conds, 2)))
    bonf = ALPHA / max(n_pairs, 1)
    for a, b in combinations(conds, 2):
        xa = sub[sub.condition == a]['value'].values
        xb = sub[sub.condition == b]['value'].values
        if len(xa) == 0 or len(xb) == 0:
            continue
        try:
            U, p = stats.mannwhitneyu(xa, xb, alternative='two-sided')
        except ValueError:
            continue
        pairs.append({'metric': metric, 'a': a, 'b': b, 'U': U, 'p': round(p, 4),
                      'p_bonferroni': round(min(p * n_pairs, 1.0), 4),
                      'sig_after_bonferroni': p * n_pairs < ALPHA})
pd.DataFrame(pairs)

## 6. Per-metric boxplots

In [ ]:
metrics = sorted(df['metric'].unique())
fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4), squeeze=False)
for ax, metric in zip(axes[0], metrics):
    sub = df[df['metric'] == metric]
    data, labels = [], []
    for cond in CONDITIONS:
        vals = sub[sub.condition == cond]['value'].values
        if len(vals):
            data.append(vals)
            labels.append(cond)
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()

## 7. Export the final comparison table

Combine descriptive + omnibus into one wide table ready for the dissertation appendix.

In [ ]:
wide = desc['median'].unstack('condition')
wide = wide.join(omnibus_df.set_index('metric')[['p', 'significant']])
wide.to_csv(REPORTS_DIR / 'comparison_summary.csv')
wide